# CS 195: Natural Language Processing
## Tool Calling

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ericmanley/s26-CS195NLP/blob/main/F7_1_ToolCalling.ipynb)


## Reminder on Fine-Tuning Applied Exploration

If you didn't demo a fine-tuning example for your applied exploration today, you can still do that during this last fortnight. I would like everyone to try some fine-tuning!

## References

- [Hugging Face chat templates](https://huggingface.co/docs/transformers/en/chat_templating)
- [Qwen2.5-0.5B-Instruct model card](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct)
- [OpenAI function calling guide](https://platform.openai.com/docs/guides/function-calling) for comparison with production-style tool calling


In [ ]:
#import sys
#!{sys.executable} -m pip install -U transformers accelerate requests


## Agentic AI

One of the buzzwords surrounding AI right now is "Agents"

In the context of language models, this usually means that the language models are connected to **tools** that allow them to interact with a larger environment

Today we'll build a small tool-calling assistant.

A tool-calling assistant does not just answer directly. It can:

1. read a user question
2. decide whether a tool is needed
3. produce a structured tool call
4. let Python run that tool
5. use the tool result to answer the user


## Imports and Device


In [4]:
import json
import re
import requests
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print('Using device:', device)


Using device: mps


## Create Tools = Write normal Python functions

Here's an example of a very simple calendar and some functions for listing and creating events.

It'd be cool to hook this up to a real calendar (Google, Outlook, etc.) as a creative synthesis project

In [5]:
import datetime

example_date = datetime.date(2026,4,27)
datetime.time()

CALENDAR = [
        { "title" : "Office hours", "date" : example_date, "start_time" : datetime.time(9,30), "duration" : 150},
        { "title" : "NLP class", "date" : example_date, "start_time" : datetime.time(12,30), "duration" : 75},
        { "title" : "Curriculum committee meeting", "date" : example_date, "start_time" : datetime.time(15), "duration" : 60}
]

def list_events(month, day, year):
   query_date = datetime.date(year,month,day)
   events = [e for e in CALENDAR if e["date"] == query_date]
   return str(events)

def create_event(title, start_datetime, duration):
    dt = datetime.datetime.fromisoformat(start_datetime)
    event_date = dt.date()
    start_time = dt.time()
    new_event = { "title" : title, "date" : event_date, "start_time" : start_time, "duration" : duration}
    CALENDAR.append(new_event)
    return new_event

print("LISTING EVENTS on 4/27/2026")
print(list_events(4,27,2026))

print("\nCREATING A NEW TEST EVENT")
print(create_event("test_event",datetime.datetime.now().isoformat(),30))

print("\nUPDATED CALENDAR")
print(CALENDAR)
    

LISTING EVENTS on 4/27/2026
[{'title': 'Office hours', 'date': datetime.date(2026, 4, 27), 'start_time': datetime.time(9, 30), 'duration': 150}, {'title': 'NLP class', 'date': datetime.date(2026, 4, 27), 'start_time': datetime.time(12, 30), 'duration': 75}, {'title': 'Curriculum committee meeting', 'date': datetime.date(2026, 4, 27), 'start_time': datetime.time(15, 0), 'duration': 60}]

CREATING A NEW TEST EVENT
{'title': 'test_event', 'date': datetime.date(2026, 4, 27), 'start_time': datetime.time(0, 16, 16, 840468), 'duration': 30}

UPDATED CALENDAR
[{'title': 'Office hours', 'date': datetime.date(2026, 4, 27), 'start_time': datetime.time(9, 30), 'duration': 150}, {'title': 'NLP class', 'date': datetime.date(2026, 4, 27), 'start_time': datetime.time(12, 30), 'duration': 75}, {'title': 'Curriculum committee meeting', 'date': datetime.date(2026, 4, 27), 'start_time': datetime.time(15, 0), 'duration': 60}, {'title': 'test_event', 'date': datetime.date(2026, 4, 27), 'start_time': datetim

## Describe the Tools for the Model

The model cannot see our Python functions directly. We need to describe:

- the tool name
- what the tool does
- what arguments it expects

In production APIs, this is usually done with JSON schema. Today we will use a simplified version of that idea.


In [6]:
tool_specs = [
    {
        'name': 'list_events',
        'description': 'List calendar events on a date.',
        'arguments': {
            'month': 'integer, required. 1=January, 2=February, etc.',
            'day' : 'integer, required. Numerical day of the month.',
            'year' : 'integer, required. Year as an integer. Example: 2026'
        }
    },
    {
        'name': 'create_event',
        'description': 'Create a new calendar event.',
        'arguments': {
            'title': 'string, required',
            'start_datetime': 'string, required. date and time in ISO format. Example: "2024-03-27T10:30:00.123456"',
            'duration': 'integer, required. Number of minutes in the event'
        }
    }
]

## Load a Small Instruct Model

We'll try Qwen2.5-0.5B again. Note that the coding pattern we use for this follows https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct?library=transformers



In [33]:
checkpoint = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(checkpoint)

model = AutoModelForCausalLM.from_pretrained(checkpoint)
model.to(device)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

In [48]:
system_prompt = f"""
You are a calendar tool-calling assistant.

Ther current datetime is {datetime.datetime.now().isoformat()}.

Choose the best tool for the user's calendar request.

Available tools: {str(tool_specs)}

Return only one JSON object. Do not include markdown or explanation.
Use this format:
{{
  "tool_name": "name_of_tool",
  "arguments": {{}}
}}
"""

user_request = "What do I have on my calendar for April 27, 2026?"

chat_messages = [
    { "role" : "system", "content" : system_prompt},
    { "role" : "user", "content" : user_request}
]

In [49]:
def generate_chat_response(messages, max_new_tokens=250):
    inputs = tokenizer.apply_chat_template(
    	messages,
    	add_generation_prompt=True,
    	tokenize=True,
    	return_dict=True,
    	return_tensors="pt",
    ).to(model.device)

    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens)
    return tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:],skip_special_tokens=True)


print("MESSAGES")
print(chat_messages)
tool_response = generate_chat_response(chat_messages)
print("NEW RESPONSE")
print(tool_response)
chat_messages.append({
    "role" : "assistant",
    "content" : tool_response
})


MESSAGES
[{'role': 'system', 'content': '\nYou are a calendar tool-calling assistant.\n\nTher current datetime is 2026-04-27T00:49:37.136775.\n\nChoose the best tool for the user\'s calendar request.\n\nAvailable tools: [{\'name\': \'list_events\', \'description\': \'List calendar events on a date.\', \'arguments\': {\'month\': \'integer, required. 1=January, 2=February, etc.\', \'day\': \'integer, required. Numerical day of the month.\', \'year\': \'integer, required. Year as an integer. Example: 2026\'}}, {\'name\': \'create_event\', \'description\': \'Create a new calendar event.\', \'arguments\': {\'title\': \'string, required\', \'start_datetime\': \'string, required. date and time in ISO format. Example: "2024-03-27T10:30:00.123456"\', \'duration\': \'integer, required. Number of minutes in the event\'}}]\n\nReturn only one JSON object. Do not include markdown or explanation.\nUse this format:\n{\n  "tool_name": "name_of_tool",\n  "arguments": {}\n}\n'}, {'role': 'user', 'content

In [50]:
import json

tool_response_dict = json.loads(tool_response)
print(tool_response_dict)


{'tool_name': 'list_events', 'arguments': {'month': 4, 'day': 27, 'year': 2026}}


## Execute the Tool Call

Now we have to actually execute the function that goes with the tool that the model wants to use.


In [51]:
if tool_response_dict["tool_name"] == "list_events":
    tool_result = list_events(**tool_response_dict["arguments"])
elif tool_response_dict["tool_name"] == "create_event":
    tool_result = create_event(**tool_response_dict["arguments"])
else:
    tool_result = "No tool executed"

print(tool_result)


[{'title': 'Office hours', 'date': datetime.date(2026, 4, 27), 'start_time': datetime.time(9, 30), 'duration': 150}, {'title': 'NLP class', 'date': datetime.date(2026, 4, 27), 'start_time': datetime.time(12, 30), 'duration': 75}, {'title': 'Curriculum committee meeting', 'date': datetime.date(2026, 4, 27), 'start_time': datetime.time(15, 0), 'duration': 60}, {'title': 'test_event', 'date': datetime.date(2026, 4, 27), 'start_time': datetime.time(0, 16, 16, 840468), 'duration': 30}]


## Use the Tool Result to Answer

The tool result may be structured data. We will send that result back to the model and ask it to write a normal answer for the user.


In [52]:
chat_messages.append({
    "role" : "system",
    "content" : f"Tool execution had the following result: {str(tool_result)}. Answer the user's original question using this information."
})

print("MESSAGES")
print(chat_messages)
final_response = generate_chat_response(chat_messages)
print("\nFINAL RESPONSE")
print(final_response)

MESSAGES
[{'role': 'system', 'content': '\nYou are a calendar tool-calling assistant.\n\nTher current datetime is 2026-04-27T00:49:37.136775.\n\nChoose the best tool for the user\'s calendar request.\n\nAvailable tools: [{\'name\': \'list_events\', \'description\': \'List calendar events on a date.\', \'arguments\': {\'month\': \'integer, required. 1=January, 2=February, etc.\', \'day\': \'integer, required. Numerical day of the month.\', \'year\': \'integer, required. Year as an integer. Example: 2026\'}}, {\'name\': \'create_event\', \'description\': \'Create a new calendar event.\', \'arguments\': {\'title\': \'string, required\', \'start_datetime\': \'string, required. date and time in ISO format. Example: "2024-03-27T10:30:00.123456"\', \'duration\': \'integer, required. Number of minutes in the event\'}}]\n\nReturn only one JSON object. Do not include markdown or explanation.\nUse this format:\n{\n  "tool_name": "name_of_tool",\n  "arguments": {}\n}\n'}, {'role': 'user', 'content

## Exercise

Try some other user prompts like 
* "What do I have on my calendar for tomorrow?"
* "add a department meeting to my calendar for May 1, 2026"
* "Am I available at 11:00am on April 27, 2026?"

## Discussion

What are the limitations of this approach with this model?

What ideas do you have to improve it?

## Discussion

What are some additional tools that you think would be useful to add to the calendar agent?

## Discussion

Think back to how we've use the course data (both for RAG and fine-tuning). Are there any cases where it would be better to retrieve information from this dataset using a tool call rather than semantic search? Is it possible to create an application that does both? How would you do it in this case?

## Applied Exploration

Add two new tools to the calendar application. Come up with a series of test instructions and evaluate the model based on how well it performed at tool selection.

## Extended Implementation Idea: Fine-Tuning Tool Choice

In the previous notebook, we fine-tuned a model by giving it many examples of the kind of response we wanted. Try doing that here, but instead of teaching the model to answer questions about the course data, teach it to choose the right tool and arguments more reliably.

If you were fine-tuning for this task, the examples should look as much as possible like the inputs the model will see at inference time. That means you would usually want to include the same `system_prompt` variable from above, since that prompt contains the tool instructions and output format.

For example, one training example might look like this:

```json
{
  "messages": [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "What do I have on my calendar for April 27, 2026?"},
    {"role": "assistant", "content": "{\"tool_name\": \"list_events\", \"arguments\": {\"month\": 4, \"day\": 27, \"year\": 2026}}"}
  ]
}
```

Another one might look like this:

```json
{
  "messages": [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "Add a department meeting to my calendar for May 1, 2026 at 2:00pm for 60 minutes."},
    {"role": "assistant", "content": "{\"tool_name\": \"create_event\", \"arguments\": {\"title\": \"department meeting\", \"start_datetime\": \"2026-05-01T14:00:00\", \"duration\": 60}}"}
  ]
}
```

You would want many examples with varied wording, such as:
- "Show me everything on my calendar for next Tuesday."
- "Do I have anything scheduled on May 3, 2026?"
- "Put office hours on my calendar tomorrow at 1:00pm for 90 minutes."
- "Schedule a project meeting for April 30, 2026 at 3:30pm for 45 minutes."
- "Add lunch with Sam on Friday at noon for an hour."



## Extended Implementation Idea: Hook it up to a real calendar

Real cloud-based calendars (Google, Outlook, etc.) have APIs that you can use to interact with the calendar. Rewrite the tools so that it works with a real calendar. Google's API is probably a little friendlier than Outlook's.